# **Understanding eos_token and pad_token in model generate config : Huggingface Transformers**

- [Text Generation](https://huggingface.co/docs/transformers/llm_tutorial?utm_source=chatgpt.com)

- [Main class](https://huggingface.co/docs/transformers/v4.57.0/en/main_classes/text_generation#transformers.GenerationMixin.generate)


**What is an eos_token ?**

- `EOS` stands for End of Sequence. It is a special token the model uses to indicate that a sequence of text has finished.

- During generation, the model predicts one token at a time, appending it to the input sequence for the next step.

- When the model outputs the `EOS` token, it signals to the generation function that the sequence should stop.

- The `EOS` token is particularly important in open-ended generation tasks where the model doesn’t know in advance how long the output should be.

- Open-ended generation means there is no predefined end boundary for the text. The model stops only when it hits `eos_token` or `max_new_tokens`.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

/Users/user/Projects/dl-llm-majors/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [ ]:
# Load model directly
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model.to(device)

#Generate an output with no eos_token and max_new_tokens set 
inputs = tokenizer("Hello, how are you?", return_tensors="pt").to(model.device)

# Generate without max_new_tokens
outputs = model.generate(
    **inputs,
    return_dict_in_generate=True,
)

generated_ids = outputs.sequences[0]
print("Tokens:", generated_ids)
print("Decoded:", tokenizer.decode(generated_ids))
print("Last token ID:", generated_ids[-1], "EOS token ID:", tokenizer.eos_token_id)
print("Number of new tokens generated: " , len(generated_ids[inputs['input_ids'].shape[-1] : ]))

'''
Output : 
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Tokens: tensor([128000,   9906,     11,   1268,    527,    499,     30,    358,   2846,
         12304,    311,   4430,    449,    499,    264,  27387,  18841,    358,
          1903,   3432,     13,   1666,    358,    574,  11689,    311,    279],
       device='cuda:0')
Decoded: <|begin_of_text|>Hello, how are you? I'm excited to share with you a fascinating discovery I made today. As I was walking to the
Last token ID: tensor(279, device='cuda:0') EOS token ID: 128009
Number of new tokens generated:  20
'''

As you can see from the above code snippet, even when you dont explicitly pass any paremeter to set the library does this for you:

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.

- The `EOS` token ID: 128009 is for the modle `llama-3.1–1b-instruct` that has been used.

- In general the model stops generating the next token whenever it sees the 128009 token.

- But here as you can see the ouput has been truncated and last token id that was generated is not equal to the `eos_token_id`.

- The reason for this abrupt termination of output even when the last token that was generated is not `eos_token_id` is the fact that the `max_new_tokens` is initialized to 20 by default.

- So, no matter weather the model generates `eos_token` or not, the generation stops after 20 tokens.

Now, will increase `max_new_tokens` and see if the model actually generated `eos_token` :

In [ ]:
inputs = tokenizer("Hello, how are you?", return_tensors="pt").to(model.device)

# Generate without max_new_tokens
outputs = model.generate(
    **inputs,
    return_dict_in_generate=True,
    max_new_tokens = 150
)

generated_ids = outputs.sequences[0]
print("Tokens:", generated_ids)
print("Decoded:", tokenizer.decode(generated_ids))
print("Last token ID:", generated_ids[-1], "EOS token ID:", tokenizer.eos_token_id)
print("Number of new tokens generated " , len(generated_ids[inputs['input_ids'].shape[-1] : ]))

'''
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Tokens: tensor([128000,   9906,     11,   1268,    527,    499,     30,    358,   2846,
          3515,    264,   2766,    315,    264,  11501,    382,     40,    574,
          2133,    311,    733,    369,    264,   1629,     11,    719,    358,
          2846,    539,   8430,    709,    311,    433,     13,    358,   2846,
         19781,     11,    358,   2846,  36366,     11,    323,    358,   2846,
          1120,    539,   8430,   1093,   7182,     13,    358,   2846,   1101,
          8430,    264,   2766,  38100,    323,  43206,    382,     40,   3077,
          1027,   4560,    311,    636,   1203,   1139,   4401,     11,    719,
           433,    596,   1120,    539,  12765,     13,    358,   2846,   6041,
           311,   2733,   1093,    358,   2846,   1120,    539,   1695,   3403,
            11,   1093,    358,   2846,    539,  13171,    315,   3815,    433,
           382,     40,   2846,   1101,   8430,   2216,  33630,    449,   7182,
           369,    539,   1694,   3025,    311,   4585,   1555,    279,   6784,
           323,   1120,   2567,   2133,     13,    358,   1440,    430,    596,
           539,   9498,     11,    719,    433,    596,   2653,    311,  22884,
           279,  62461,    311,   1120,   3041,    709,    382,     40,   2846,
          6041,    311,   5895,    422,    358,   3358,   3596,    387,   3025,
           311,   1629,   1578,     13], device='cuda:0')
Decoded: <|begin_of_text|>Hello, how are you? I'm having a bit of a crisis.

I was going to go for a run, but I'm not feeling up to it. I'm tired, I'm sore, and I'm just not feeling like myself. I'm also feeling a bit anxious and overwhelmed.

I've been trying to get back into running, but it's just not happening. I'm starting to feel like I'm just not good enough, like I'm not capable of doing it.

I'm also feeling really frustrated with myself for not being able to push through the pain and just keep going. I know that's not healthy, but it's hard to resist the temptation to just give up.

I'm starting to wonder if I'll ever be able to run again.
Last token ID: tensor(13, device='cuda:0') EOS token ID: 128009
Number of new tokens generated  150
'''

As you can see at `max_new_token=150` the models still did not generate `eos_token_id`, it has stopped its generation solely due to the token limit.

Lets try `max_new_token = 300`

In [ ]:
inputs = tokenizer("Hello, how are you?", return_tensors="pt").to(model.device)

# Generate without max_new_tokens
outputs = model.generate(
    **inputs,
    return_dict_in_generate=True,
    max_new_tokens = 300
)

generated_ids = outputs.sequences[0]
print("Tokens:", generated_ids)
print("Decoded:", tokenizer.decode(generated_ids))
print("Last token ID:", generated_ids[-1], "EOS token ID:", tokenizer.eos_token_id)
print("Number of new tokens generated " , len(generated_ids[inputs['input_ids'].shape[-1] : ]))

'''
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Tokens: tensor([128000,   9906,     11,   1268,    527,    499,     30,    358,   1097,
           779,  12304,    311,   4430,    420,    449,    499,     11,    719,
           358,   1205,    311,   8985,    499,     11,    358,   1097,    539,
           264,   6721,    304,    420,   2115,     13,    358,   1097,   1120,
           264,  22999,   1732,    889,    374,  24450,    311,   4048,    323,
         13488,    502,   2574,    382,   4516,     11,    358,    574,  20910,
            11,   1148,    374,    279,   1455,   7185,   3245,    499,   3077,
          9687,   6051,     30,    358,   2846,   2744,   3411,    369,    502,
          6848,    323,  20343,     13,    358,   3077,   1027,   5403,    264,
          2763,    315,   6603,  31445,    323,    358,   2846,  22999,    311,
          6865,    922,   1148,    499,   3077,   1027,    709,    311,    382,
         13699,     11,    358,    574,  20910,    422,    499,   1436,   7079,
          1063,   6603,    430,    358,   2643,   4774,     30,    358,   2846,
          2744,   3411,    369,    502,  12283,    323,  13650,    311,  13488,
            13,    358,   3077,   1027,   2768,    264,   2763,    315,   8198,
           323,   5557,  26743,  31445,     11,    323,    358,   2846,   2216,
          8173,    304,   6975,    810,    922,    279,  19801,    315,   1521,
          1403,   5151,    382,  81586,     11,    358,    574,  20910,    422,
           499,   1436,   7079,   1063,  55346,    430,    358,   2643,   4774,
            30,    358,   2846,   2744,   3411,    369,    502,   5039,    311,
          9020,    311,    323,    358,   2846,   2216,   8173,    304,   6975,
           810,    922,    279,   5652,  26006,    304,   8198,    323,   5557,
           382,  12947,    369,   4737,    279,    892,    311,   6369,    449,
           757,     11,    358,   2216,  15763,    433,      0,    358,   2846,
          3411,   4741,    311,  11011,    701,  11555,    323,  19075,    382,
         14809,  24886,    345,     58,   7927,   4076,     60, 128009],
       device='cuda:0')
Decoded: <|begin_of_text|>Hello, how are you? I am so excited to share this with you, but I need to warn you, I am not a professional in this field. I am just a curious person who is eager to learn and explore new things.

So, I was wondering, what is the most interesting thing you've learned recently? I'm always looking for new ideas and inspiration. I've been reading a lot of books lately and I'm curious to hear about what you've been up to.

Also, I was wondering if you could recommend some books that I might enjoy? I'm always looking for new authors and topics to explore. I've been following a lot of science and technology blogs lately, and I'm really interested in learning more about the intersection of these two fields.

Lastly, I was wondering if you could recommend some podcasts that I might enjoy? I'm always looking for new shows to listen to and I'm really interested in learning more about the latest developments in science and technology.

Thanks for taking the time to chat with me, I really appreciate it! I'm looking forward to hearing your thoughts and recommendations.

Best regards,
[Your Name]<|eot_id|>
Last token ID: tensor(128009, device='cuda:0') EOS token ID: 128009
Number of new tokens generated  226
'''

Tada!!
The model has finally genrated `eos_token_id = 128009` as it `226th` token.


So, clearly the model here has STOPPED generating next tokens as it has encountered the `eos_token_id` at `226th` token which is less the than the permitted `max_new_tokens=300`.



**Hence we can conclude the following:**

The `model.generate()` stops generating next token in any one of the following scenarios:

1. When then model encounters `eos_token_id` before it maxes out its generated tokens to value set for `max_new_tokens`.

2. or, when the model generated tokens count hits the `max_new_tokens` and no `eos_token_id` has been generated in the sequence.



**2. What role does the max_new_tokens play in pushing the model towards eos_token — should you care about it?**

As, seen from the above examples, increasing the `max_new_tokens` value allows the model to freely generate the response and end cleanly by itself by generating `eos_token_id`.
